# 04 — Feature Engineering & Representation

The statistical stage identifies candidate signals. This notebook turns those signals into modeling features.

The main questions are:

- Which behavioral measures are most useful?
- Should behavior be represented as counts, proportions or normalized measures?
- How should high-cardinality categorical variables be represented?
- Which variables should be excluded because they are unavailable at the prediction point or potentially leakage-prone?

## Load data

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd

DATA_PATH = Path("../data/synthetic/synthetic_early_modeling_base.csv")
df = pd.read_csv(DATA_PATH)

df.shape

## 1. Prediction-point features

Only information observed by approximately one-third of the original loan lifecycle is allowed into the feature set.

Full-lifecycle outcome-generating variables are kept separate from candidate predictors.

In [ ]:
TARGET = "is_good_or_bad"

base_features = [
    "frequency_name",
    "product_group",
    "sector",
    "region",
    "original_loan_duration_days",
    "original_no_of_installments",
    "disbursed_amount",
    "installment_amount",
    "interest_rate",
    "prior_loan_count",
    "early_missed_installment_count",
    "early_max_consecutive_missed",
    "early_max_overdue_days",
    "early_total_overdue_days",
    "early_recovery_delay_cycles",
    "prediction_installment",
]

model_df = df[base_features + [TARGET]].copy()

model_df.head()

## 2. Behavioral representations

The same underlying behavior can be represented in more than one way.

Here the candidate representations include:

- raw missed-installment count;
- missed-installment proportion;
- raw overdue duration;
- overdue duration in repayment cycles;
- overdue amount proxy;
- relative overdue burden;
- maximum consecutive misses;
- recovery delay.

In [ ]:
freq_days = model_df["frequency_name"].map({
    "Weekly": 7,
    "Bi-weekly": 14,
    "Monthly": 28,
})

model_df["pass_due_cycle_ratio"] = (
    model_df["early_max_overdue_days"] / freq_days
)

model_df["overdue_installment_equivalent"] = (
    model_df["early_total_overdue_days"] / freq_days
)

model_df["missed_installment_proportion"] = (
    model_df["early_missed_installment_count"]
    / model_df["prediction_installment"].clip(lower=1)
)

model_df["overdue_amount_proxy"] = (
    model_df["early_missed_installment_count"]
    * model_df["installment_amount"]
)

model_df["overdue_proportion"] = (
    model_df["overdue_amount_proxy"]
    / (
        model_df["prediction_installment"].clip(lower=1)
        * model_df["installment_amount"]
    )
).clip(0, 1)

model_df[[
    "early_missed_installment_count",
    "missed_installment_proportion",
    "early_max_overdue_days",
    "pass_due_cycle_ratio",
    "early_max_consecutive_missed",
    "early_recovery_delay_cycles",
    "overdue_proportion"
]].head()

## 3. Compare raw and normalized behavior

Counts and calendar durations are easy to interpret, but they can be affected by the number of installments and repayment frequency.

The normalized versions are retained as explicit candidates rather than assuming one representation is automatically better.

In [ ]:
representation_summary = model_df[[
    "early_missed_installment_count",
    "missed_installment_proportion",
    "early_max_overdue_days",
    "pass_due_cycle_ratio",
    "early_total_overdue_days",
    "overdue_installment_equivalent",
    "overdue_proportion",
]].describe().T

representation_summary

## 4. Missingness as information

Missing values are inspected before choosing an encoding strategy.

A missing field can mean:

- genuinely unavailable information;
- no recorded activity;
- an operational data gap.

Those cases should not automatically be treated as equivalent.

In [ ]:
missing_summary = (
    model_df.isna().mean()
    .mul(100)
    .sort_values(ascending=False)
    .rename("missing_pct")
    .to_frame()
)

missing_summary

## 5. Categorical representation

The original work explored high-cardinality product and geographic variables and eventually considered grouping/merging sparse categories.

The public version keeps the raw categories first so the dimensionality problem can be measured explicitly.

In [ ]:
categorical_cols = [
    "frequency_name",
    "product_group",
    "sector",
    "region",
]

category_cardinality = (
    model_df[categorical_cols]
    .nunique(dropna=False)
    .sort_values(ascending=False)
    .rename("unique_values")
    .to_frame()
)

category_cardinality

## 6. One-hot representation

One-hot encoding preserves category detail but can create a wide sparse feature matrix.

This is useful as a baseline representation before testing more compact grouping strategies.

In [ ]:
encoded = pd.get_dummies(
    model_df[categorical_cols],
    drop_first=True,
    dtype=np.int8
)

print("Categorical columns:", len(categorical_cols))
print("Encoded columns:", encoded.shape[1])

## 7. Sparse-category grouping

High-cardinality categorical variables can become problematic when many categories have very few observations.

A simple public demonstration is to group infrequent levels into `Other`.

The threshold is a modeling choice and should be tuned on the training data in a real experiment.

In [ ]:
def group_rare_categories(series, min_rate=0.01):
    freq = series.value_counts(normalize=True)
    keep = freq[freq >= min_rate].index
    return series.where(series.isin(keep), "Other")

grouped = model_df.copy()

for col in ["product_group", "sector", "region"]:
    grouped[col] = group_rare_categories(grouped[col])

grouped[["product_group", "sector", "region"]].head()

## 8. Feature-screening table

Keep feature engineering separate from model fitting so each representation can be compared consistently.

In [ ]:
candidate_features = [
    "early_missed_installment_count",
    "missed_installment_proportion",
    "early_max_overdue_days",
    "pass_due_cycle_ratio",
    "early_total_overdue_days",
    "overdue_installment_equivalent",
    "early_max_consecutive_missed",
    "early_recovery_delay_cycles",
    "overdue_proportion",
    "disbursed_amount",
    "installment_amount",
    "interest_rate",
    "prior_loan_count",
]

feature_screen = (
    model_df[candidate_features]
    .agg(["count", "mean", "std", "min", "max"])
    .T
)

feature_screen

## 9. Modeling set

The final feature matrix will be selected only after comparing representations and checking:

- prediction-time availability;
- leakage risk;
- missingness;
- dimensionality;
- interpretability;
- predictive value.

In [ ]:
numeric_features = [
    "missed_installment_proportion",
    "pass_due_cycle_ratio",
    "overdue_installment_equivalent",
    "overdue_proportion",
    "early_max_consecutive_missed",
    "early_recovery_delay_cycles",
    "disbursed_amount",
    "installment_amount",
    "interest_rate",
    "prior_loan_count",
]

categorical_features = [
    "frequency_name",
    "product_group",
    "sector",
    "region",
]

X = model_df[numeric_features + categorical_features].copy()
y = model_df[TARGET].copy()

X.shape, y.mean()

## Takeaway

The key feature-engineering decisions are treated as empirical questions rather than fixed preprocessing rules:

- count vs proportion;
- raw duration vs normalized duration;
- detailed categories vs grouped categories;
- observed missingness vs imputation.

The next stage will compare model families using these alternative representations.